# 🚀 GOD LEVEL EDA AUTOMATION TOOLKIT.

Is notebook mein hum ek powerful **EDA (Exploratory Data Analysis) automation system** bana rahe hain.

**EDA kya hota hai?**
Jab bhi hume koi naya dataset milta hai, pehle hum usse acchi tarah samajhte hain —
kitne rows hain, koi missing data toh nahi, koi outlier toh nahi — yahi sab kaam EDA mein hota hai.

Har function ke upar explanation diya gaya hai taki samajhne mein easy ho. 🙂


## 📦 Step 1: Libraries Import Karna

Sabse pehle hum teen important libraries import karte hain:

- **pandas** → Data ko table (DataFrame) format mein handle karne ke liye
- **numpy** → Mathematical aur numerical calculations ke liye
- **warnings** → Unnecessary warning messages ko hide karne ke liye, taaki output clean rahe


In [1]:
import pandas as pd        # pandas → DataFrame banane aur data manipulate karne ke liye
import numpy as np         # numpy → numbers pe calculations karne ke liye (mean, std, etc.)
import warnings            # warnings → warning messages ko control karne ke liye

warnings.filterwarnings("ignore")  # sabhi unnecessary warnings ko ignore/hide kar do


## 📌 Step 2: Basic Info Function

Ye function DataFrame ke baare mein **basic jaankari** deta hai:

- **Shape** → Kitne rows aur kitne columns hain
- **Columns** → Saare column names ki list
- **Data Types** → Har column ka type kya hai (number hai ya text)

Jab bhi naya dataset aaye, sabse pehle ye function chalao — isse dataset ka structure samajh aata hai.


In [2]:
def basic_info(df):                                        # df naam ka DataFrame input mein lega
    print("📌 Shape:", df.shape)                          # df.shape → (rows, columns) ka tuple print karta hai
    print("\n📌 Columns:", df.columns.tolist())           # df.columns → column names, tolist() se list mein convert
    print("\n📌 Data Types:\n", df.dtypes)                # df.dtypes → har column ka data type dikhata hai


## ❓ Step 3: Missing Values Checker

Dataset mein aksar kuch values missing (blank/NaN) hoti hain — ye function unhe dhundta hai.

**Ye function kya return karta hai?**
- Har column mein kitni values missing hain (count)
- Aur percentage mein kitni missing hain
- Sabse zyada missing wale columns upar dikhate hain (sorted)

Isse pata chalta hai ki kaunse columns ko fix karna padega.


In [3]:
def missing_values(df):                                              # df DataFrame input mein lega
    missing = df.isnull().sum()                                      # isnull() → har cell check karo missing hai ya nahi, sum() → total count nikalo
    percent = (missing / len(df)) * 100                             # har column ka missing percentage nikalo (count / total rows * 100)
    result = pd.DataFrame({                                          # ek nayi table (DataFrame) banao dono cheezein ek saath dikhane ke liye
        "Missing Count": missing,                                    # pehla column → missing values ki count
        "Percentage": percent                                        # doosra column → percentage mein missing
    })
    return result.sort_values(by="Percentage", ascending=False)      # Percentage ke hisaab se sort karo — sabse zyada missing wala upar


## 🔁 Step 4: Duplicate Rows Checker

Kabhi kabhi dataset mein ek hi row baar baar repeat ho jaati hai — ise **duplicate** kehte hain.

Ye function sirf ek number return karta hai — **kitne duplicate rows hain**.

Agar count > 0 hai, toh data ko clean karna padega (`drop_duplicates()` use karke).


In [4]:
def duplicate_info(df):                  # df DataFrame input mein lega
    return df.duplicated().sum()         # duplicated() → har row check karo duplicate hai ya nahi, sum() → total duplicate rows count karo


## 📊 Step 5: Statistical Summary

Ye function dataset ke **saare numeric columns ka statistical overview** deta hai.

**`describe()` se kya milta hai?**
- count → kitni values hain
- mean → average
- std → standard deviation (values kitni spread hain)
- min / max → sabse chhoti aur badi value
- 25%, 50%, 75% → quartile values

Ek hi function mein poora data ka number-wise mood samajh aata hai! 😎


In [5]:
def stats_summary(df):       # df DataFrame input mein lega
    return df.describe()     # describe() → sabhi numeric columns ka count, mean, std, min, max, quartiles ek saath dikhata hai


## 🚨 Step 6: Outlier Detection (IQR Method)

**Outlier** woh values hain jo baaki data se bahut zyada alag hoti hain — bahut badi ya bahut choti.
Jaise agar sab logo ki age 20-40 hai aur ek ki 200 hai — woh outlier hai.

**IQR (Interquartile Range) Method kaise kaam karta hai?**
1. Q1 (25th percentile) aur Q3 (75th percentile) nikalo
2. IQR = Q3 - Q1
3. Lower limit = Q1 - 1.5 × IQR
4. Upper limit = Q3 + 1.5 × IQR
5. Jo values in limits ke bahar hain → woh outliers hain

Ye function har numeric column ke liye outliers ki count return karta hai.


In [6]:
def outlier_check(df):                                                          # df DataFrame input mein lega
    outliers = {}                                                                # ek empty dictionary banao jisme results store honge
    for col in df.select_dtypes(include=np.number).columns:                     # sirf numeric columns pe loop chalo (text columns skip)
        Q1 = df[col].quantile(0.25)                                             # Q1 → 25th percentile value nikalo (niche se 25% data yahan tak)
        Q3 = df[col].quantile(0.75)                                             # Q3 → 75th percentile value nikalo (niche se 75% data yahan tak)
        IQR = Q3 - Q1                                                           # IQR = Q3 minus Q1 → middle 50% data ka range
        lower = Q1 - 1.5 * IQR                                                  # lower bound → is se niche ki values outlier hain
        upper = Q3 + 1.5 * IQR                                                  # upper bound → is se upar ki values outlier hain
        count = df[(df[col] < lower) | (df[col] > upper)].shape[0]             # lower se niche ya upper se upar wali rows count karo
        outliers[col] = count                                                    # us column ka outlier count dictionary mein save karo
    return outliers                                                              # saare columns ke outlier counts return karo


## 🏷️ Step 7: Categorical Summary

**Categorical columns** woh hote hain jisme text hota hai — jaise Gender (Male/Female), City (Mumbai/Delhi), etc.

Ye function har categorical column ke liye dikhata hai:
- Kaun si values hain
- Har value kitni baar aayi hai (frequency)

Isse pata chalta hai ki koi category dominate toh nahi kar rahi ya koi spelling mistake toh nahi.


In [7]:
def categorical_summary(df):                              # df DataFrame input mein lega
    cat_cols = df.select_dtypes(include="object").columns  # sirf text/string wale columns select karo ("object" = text type)
    for col in cat_cols:                                   # har categorical column pe loop chalo
        print(f"\n📊 Column: {col}")                      # column ka naam print karo
        print(df[col].value_counts())                      # value_counts() → har unique value kitni baar aayi, count ke saath dikhao


## 🔢 Step 8: Encoding Helper (Text → Numbers)

Machine Learning models sirf numbers samajhte hain, text nahi.
Isliye text columns ko numbers mein convert karna padta hai — ise **Encoding** kehte hain.

**One-Hot Encoding kya hai?**
Jaise `Gender` column mein Male/Female hai:
- Male → [1, 0]
- Female → [0, 1]

`drop_first=True` → ek column drop karta hai redundancy avoid karne ke liye (Male column raha, Female drop ho gayi — sirf Male se hi dono pata chal jaata hai).


In [8]:
def encoding_helper(df):                            # df DataFrame input mein lega
    return pd.get_dummies(df, drop_first=True)      # get_dummies() → sabhi text columns ko one-hot encoding mein convert karo, drop_first=True → pehli category ka column drop karo (redundancy avoid karne ke liye)


## ⚖️ Step 9: Scaling Helper (Values Ko Normalize Karna)

Kabhi kabhi ek column mein values 0-1 hoti hain aur doosre mein 1000-100000 — itna difference model ke liye problem create karta hai.

**StandardScaler** sab values ko ek common scale pe le aata hai:
- Mean (average) = 0 ho jaata hai
- Standard Deviation = 1 ho jaati hai

Isse sabhi numeric columns equal importance lete hain model mein.


In [11]:
from sklearn.preprocessing import StandardScaler    # sklearn se StandardScaler import karo

def scaling_helper(df):                                              # df DataFrame input mein lega
    scaler = StandardScaler()                                        # StandardScaler ka object banao
    num_cols = df.select_dtypes(include=np.number).columns          # sirf numeric columns select karo (text columns skip)
    df[num_cols] = scaler.fit_transform(df[num_cols])               # fit_transform() → scaler ko data pe fit karo aur transform kar do (mean=0, std=1)
    return df                                                        # scaled DataFrame return karo


## 🚀 Step 10: MASTER FUNCTION — full_eda()

Ye sabka **Boss Function** hai! 👑

Sirf ek function call karo — `full_eda(df)` — aur ye apne aap:
1. Basic info dikhayega
2. Missing values check karega
3. Duplicates dhundhega
4. Statistical summary dega
5. Outliers detect karega
6. Categorical data summarize karega

Ek function → Poora EDA complete! 🎯


In [ ]:
def full_eda(df):                                         # df DataFrame input mein lega
    print("🚀 BASIC INFO")                               # heading print karo
    basic_info(df)                                        # basic_info() function call karo → shape, columns, dtypes dikhao

    print("\n🚀 MISSING VALUES")                         # heading print karo
    print(missing_values(df))                             # missing_values() call karo → missing count aur percentage dikhao

    print("\n🚀 DUPLICATES:", duplicate_info(df))        # duplicate_info() call karo → total duplicate rows ki count dikhao

    print("\n🚀 STATS SUMMARY")                          # heading print karo
    print(stats_summary(df))                              # stats_summary() call karo → mean, std, min, max etc. dikhao

    print("\n🚀 OUTLIERS")                               # heading print karo
    print(outlier_check(df))                              # outlier_check() call karo → har column ke outliers count dikhao

    print("\n🚀 CATEGORICAL DATA")                       # heading print karo
    categorical_summary(df)                               # categorical_summary() call karo → text columns ki value frequency dikhao


## ▶️ Step 11: Apna Data Load Karo aur Run Karo!

Ab bas apna CSV file load karo aur `full_eda(df)` ek baar chalao — poora analysis ho jaayega!

**Usage:**
```python
df = pd.read_csv("apna_data.csv")   # apni file ka naam yahan likho
full_eda(df)                         # ek line mein poora EDA!
```


In [ ]:
# Neeche apna CSV file ka path likho aur dono lines uncomment karo (# hata do)

# df = pd.read_csv("apna_data.csv")   # apna dataset load karo CSV file se
# full_eda(df)                         # ek baar ye chalao — poora EDA automatic ho jaayega!
